# Nernst equation and Debye–Hückel screening

*physchem · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/physchem/nernst_debye_huckel)

The voltage of a Daniell cell, Zn | Zn²⁺ ‖ Cu²⁺ | Cu, follows the Nernst equation, computed here as one $(T, Q)$ grid of temperatures and concentration ratios. Real ions are less active than their concentration suggests: each one wears a cloud of opposite charge of thickness $\kappa^{-1}$. The linearised Poisson–Boltzmann equation around an ion is solved on a radial grid as a batched linear system, matches the analytic Yukawa potential, and reproduces the Debye–Hückel activity coefficients that shift the EMF.

**Vector form**

$$E = E^\circ - \frac{RT}{nF}\ln\frac{a_{\mathrm{Zn^{2+}}}}{a_{\mathrm{Cu^{2+}}}},\qquad a = \gamma\,m,\qquad \log_{10}\gamma = -\frac{A z^2\sqrt I}{1 + B a\sqrt I},\qquad \frac{1}{r^2}\frac{d}{dr}\Big(r^2\frac{d\varphi}{dr}\Big) = \kappa^2\varphi,\qquad \kappa^2 = \frac{2N_Ae^2 I}{\varepsilon_0\varepsilon_r kT}$$

**Index (tensor) form**

$$E_{tq} = E^\circ - s_t\,\log_{10}Q_q,\quad s_t = \frac{\ln 10\,R\,T_t}{nF} \qquad \log_{10}\gamma_c = -\frac{A z^2\sqrt{I_c}}{1 + B a\sqrt{I_c}} \qquad A_{cij}\,u_{cj} = b_{ci},\quad \varphi_{ci} = u_{ci}/r_{ci} \qquad \varphi^{Y}_{ci} = \frac{e^{\kappa_c a}}{1+\kappa_c a}\,\frac{e^{-\kappa_c r_{ci}}}{r_{ci}}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $T_t,\ \log_{10}Q_q$ | `temperatures(), log_ratios()` | (T,), (Q,) | a batch of cell temperatures and reaction quotients |
| $E_{tq}$ | `E0 - einsum('t,q->tq', nernst_slope(T, n), log_Q)` | (T, Q) | no index summed: every slope meets every quotient |
| $I_c,\ \kappa_c$ | `ionic_strength(m), kappa(I, T)` | (C,) | ionic strength $I = mz^2$ of the swept ZnSO₄ half-cell and the inverse Debye length |
| $\log_{10}\gamma_c$ | `log10_gamma_extended(z, I, A, B, a)` | (C,) | extended law; $a \to 0$ gives the limiting law $-Az^2\sqrt I$ |
| $r_{ci}$ | `radial_grids(kappa, a, M)` | (C, M) | grid from the ion surface $a$ out to $a + 12\,\kappa_c^{-1}$, one per concentration |
| $A_{cij},\ b_{ci}$ | `pb_system(h, kappa, a, M)` | (C, M, M), (C, M) | $u^{\prime\prime} - \kappa^2u = 0$ with $u = r\varphi$; row 0 is Gauss's law at $r = a$ |
| $A_{cij}u_{cj} = b_{ci}$ | `torch.linalg.solve(A, b)` | (C, M) | repeated $j$ ⇒ summed; one dense solve per concentration, batched |
| $\ln\gamma_c$ | `0.5 * z**2 * l_B * (phi[:, 0] - 1/a)` | (C,) | energy of the ion in its own cloud: recovers the extended Debye–Hückel law |

In the Daniell cell zinc is oxidised and copper ions are reduced, $\mathrm{Zn + Cu^{2+} \to Zn^{2+} + Cu}$, with $n = 2$ electrons and $E^\circ \approx 1.10$ V (held fixed here, since its own temperature change is small). Every tenfold change of $Q$ moves the EMF by $\ln 10\,RT/nF$, which is 29.6 mV at 25 °C, and the slope grows in proportion to $T$. The Nernst equation uses activities $a = \gamma m$, not molalities. Debye and Hückel computed $\gamma$ by letting the other ions arrange themselves in the potential $\varphi$ of a central ion with Boltzmann weights, and linearising: $\nabla^2\varphi = \kappa^2\varphi$. Its solution is the screened Coulomb (Yukawa) potential, which falls off over the Debye length $\kappa^{-1} \approx 0.304\,\mathrm{nm}/\sqrt{I}$ in water at 25 °C. The numerical solver writes the radial equation for $u = r\varphi$ as a tridiagonal system, with Gauss's law at the ion surface and $\varphi = 0$ twelve Debye lengths out, and agrees with the analytic form to about $10^{-4}$. The ion's energy in the potential of its own cloud is $kT\ln\gamma$, which gives exactly the extended law; for $I \to 0$ it becomes the limiting law $\log_{10}\gamma = -Az^2\sqrt I$ with $A \approx 0.51$. Each half-cell is a separate 2:2 sulfate solution ($I = 4m$), so changing $m(\mathrm{Zn^{2+}})$ changes $\gamma_{Zn}$ while $\gamma_{Cu}$ stays put, and the EMF moves away from the ideal Nernst value by several millivolts. Single-ion activity coefficients cannot be measured on their own (only the mean $\gamma_\pm$ can), and above $I \approx 0.01$ ion pairing and specific interactions in 2:2 sulfates make even the extended law qualitative. The ion size $a \approx 0.5$ nm is a typical hydrated-ion value.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/physchem/nernst_debye_huckel/main.py
"""Main loop: Nernst grid over (T, Q) → Debye–Hückel activities → EMF → Poisson–Boltzmann ion clouds → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    E0, n, z, a = params["E0"], tensors.N_ELECTRONS, tensors.Z_ION, params["a_ion"]
    T = torch.tensor(params["T_C"] + 273.15, device=dev)  # the working temperature (K)

    # 1) ideal Nernst: EMF for every temperature t and reaction quotient q at once
    T_batch = tensors.temperatures(dev)  # (T,)
    log_Q = tensors.log_ratios(dev)  # (Q,)
    E_tq = equation.nernst(E0, T_batch, log_Q, n)  # (T, Q)

    # 2) Debye–Hückel: the Zn²⁺ half-cell molality is swept, the Cu²⁺ half-cell is fixed
    m_zn = tensors.zinc_molalities(60, dev)  # (C,)
    m_cu = torch.tensor(params["m_Cu"], device=dev)
    I_zn, I_cu = tensors.ionic_strength(m_zn), tensors.ionic_strength(m_cu)
    A, B = equation.debye_A(T), equation.debye_B(T)
    lg_lim_zn, lg_lim_cu = equation.log10_gamma_limiting(z, I_zn, A), equation.log10_gamma_limiting(z, I_cu, A)
    lg_ext_zn, lg_ext_cu = (equation.log10_gamma_extended(z, ion, A, B, a) for ion in (I_zn, I_cu))
    E_ideal = equation.emf(E0, T, m_zn, m_cu, 0.0, 0.0, n)  # (C,)
    E_lim = equation.emf(E0, T, m_zn, m_cu, lg_lim_zn, lg_lim_cu, n)
    E_ext = equation.emf(E0, T, m_zn, m_cu, lg_ext_zn, lg_ext_cu, n)

    # 3) the ion cloud: linearised Poisson–Boltzmann around a Zn²⁺ ion, one radial grid per concentration
    kap = equation.kappa(I_zn, T)  # (C,) nm⁻¹
    M = 120 if fast_mode() else params["n_points"]
    r, h = tensors.radial_grids(kap, a, M)  # (C, M), (C,)
    phi = solve.solve_pb(r, h, kap, a)  # (C, M) in units of q/4πε₀ε_r (nm⁻¹)
    phi_exact = equation.yukawa(r, kap, a)
    l_B = equation.bjerrum_length(T)
    lg_pb = equation.ln_gamma_from_potential(phi[:, 0], a, z, l_B) / equation.LN10  # (C,) log₁₀ γ from the cloud

    s = equation.nernst_slope(T, n)
    print(f"Daniell cell Zn|Zn²⁺||Cu²⁺|Cu   E° = {E0:.3f} V   n = {n}   T = {params['T_C']:.0f} °C")
    print(f"Nernst slope ln10·RT/nF = {1e3 * s.item():.2f} mV per decade of Q")
    print("E at Q = 0.01, 1, 100:  " + "   ".join(
        f"{round(Tk.item() - 273.15):d} °C: " + ", ".join(f"{E_tq[t, q].item():.4f}" for q in (20, 40, 60))
        for t, Tk in enumerate(T_batch)
    ))  # fmt: skip
    print(f"ε_r(water) = {equation.permittivity_water(T).item():.2f}   l_B = {l_B.item():.3f} nm   "
          f"A = {A.item():.4f} (kg/mol)^½   B = {B.item():.3f} nm⁻¹ (kg/mol)^½")  # fmt: skip
    print(f"{'m(Zn²⁺)':>9} {'I':>7} {'κ⁻¹ (nm)':>9} {'γ lim':>7} {'γ ext':>7} {'γ PB':>7} {'E ideal':>8} {'E ext':>8}")
    for c in range(0, len(m_zn), 12):
        print(
            f"{m_zn[c].item():9.1e} {I_zn[c].item():7.4f} {1 / kap[c].item():9.3f} {10 ** lg_lim_zn[c].item():7.4f} "
            f"{10 ** lg_ext_zn[c].item():7.4f} {10 ** lg_pb[c].item():7.4f} "
            f"{E_ideal[c].item():8.4f} {E_ext[c].item():8.4f}"
        )
    err = ((phi - phi_exact).abs() / phi_exact.abs().max(1, keepdim=True).values).max().item()
    print(f"Poisson–Boltzmann on {M}-point grids vs Yukawa: max error {err:.1e} of the surface potential")
    print(
        f"γ from the PB cloud vs extended Debye–Hückel: max |Δ log₁₀γ| = {(lg_pb - lg_ext_zn).abs().max().item():.1e}"
    )

    volts = equation.potential_scale(T, z)
    return plot.animate(
        log_Q, T_batch, E_tq, m_zn, m_cu, E_ideal, E_lim, E_ext, I_zn, lg_pb, A, B, a, z, T, r, phi, kap, volts,
        out_dir / "nernst_debye_huckel.gif",
    )  # fmt: skip


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/physchem/nernst_debye_huckel/equation.py
r"""Physics → tensor translation.

Nernst equation for the Daniell cell Zn | Zn²⁺ || Cu²⁺ | Cu  (Cu²⁺ + Zn → Cu + Zn²⁺, n = 2 electrons):

    E_{tq} = E° − (ln 10 · R T_t / n F) · log₁₀ Q_q,          Q = a(Zn²⁺) / a(Cu²⁺),   a = γ m

Debye–Hückel: an ion is surrounded by a cloud of opposite charge of thickness κ⁻¹, which lowers its activity:

    κ² = 2 N_A e² I / (ε₀ ε_r k T)                      (I = ½ Σ_k m_k z_k², the ionic strength)
    log₁₀ γ = −A z² √I                     (limiting law, I → 0)
    log₁₀ γ = −A z² √I / (1 + B a √I)      (extended law, ions of radius a)

The cloud comes from the linearised Poisson–Boltzmann equation around one ion of charge q and radius a:

    (1/r²) d/dr (r² dφ/dr) = κ² φ     ⇔     u'' = κ² u   with u = r φ

On a radial grid r_ci (c = concentration, i = grid point) this is a linear system for each c,

    A_{cij} u_{cj} = b_{ci}        (j summed)

whose solution is compared with the analytic screened-Coulomb (Yukawa) potential
φ = (q / 4π ε₀ ε_r) · e^{κa}/(1 + κa) · e^{−κr}/r.
"""

import math

import torch

from .tensors import E_CHARGE, EPS0, K_B, N_A, R_GAS, F

LN10 = math.log(10.0)


# ---------------------------------------------------------------- Nernst


def nernst_slope(T: torch.Tensor, n: int) -> torch.Tensor:
    """ln 10 · R T / n F: the EMF change per decade of Q (29.6 mV at 25 °C for n = 2)   (same shape as T)."""
    return LN10 * R_GAS * T / (n * F)


def nernst(E0: float, T: torch.Tensor, log10_Q: torch.Tensor, n: int) -> torch.Tensor:
    """E_tq = E° − s_t log₁₀Q_q   (T, Q).   No index summed: an outer product of slopes and log ratios."""
    return E0 - torch.einsum("t,q->tq", nernst_slope(T, n), log10_Q)


def emf(E0: float, T: float, m_zn, m_cu, log10_gamma_zn, log10_gamma_cu, n: int) -> torch.Tensor:
    """E = E° − s [log₁₀(m_Zn/m_Cu) + log₁₀γ_Zn − log₁₀γ_Cu]: Nernst with activities a = γ m."""
    s = nernst_slope(torch.as_tensor(T), n)
    return E0 - s * (torch.log10(m_zn / m_cu) + log10_gamma_zn - log10_gamma_cu)


# ---------------------------------------------------------------- Debye–Hückel


def permittivity_water(T: torch.Tensor) -> torch.Tensor:
    """ε_r of water (Malmberg & Maryott fit, t in °C): 87.74 − 0.40008 t + 9.398·10⁻⁴ t² − 1.41·10⁻⁶ t³."""
    t = T - 273.15
    return 87.74 - 0.40008 * t + 9.398e-4 * t**2 - 1.41e-6 * t**3


def bjerrum_length(T: torch.Tensor) -> torch.Tensor:
    """l_B = e² / (4π ε₀ ε_r k T): the distance at which two unit charges interact with energy kT (nm)."""
    return E_CHARGE**2 / (4 * math.pi * EPS0 * permittivity_water(T) * K_B * T) * 1e9


def kappa(ionic: torch.Tensor, T: torch.Tensor) -> torch.Tensor:
    """Inverse Debye length κ = √(2 N_A e² I · 1000 / ε₀ ε_r k T) in nm⁻¹.

    I is the ionic strength in mol/kg ≈ mol/L; the factor 1000 turns it into mol per m³.
    """
    k2 = 2 * N_A * E_CHARGE**2 * ionic * 1000 / (EPS0 * permittivity_water(T) * K_B * T)  # m⁻²
    return torch.sqrt(k2) * 1e-9


def debye_A(T: torch.Tensor) -> torch.Tensor:
    """A = l_B (κ/√I) / (2 ln 10): ≈ 0.51 (kg/mol)^½ for water at 25 °C."""
    return bjerrum_length(T) * kappa(torch.ones_like(T), T) / (2 * LN10)


def debye_B(T: torch.Tensor) -> torch.Tensor:
    """B = κ/√I: ≈ 3.29 nm⁻¹ (kg/mol)^½ at 25 °C, so κ⁻¹ ≈ 0.304 nm / √I."""
    return kappa(torch.ones_like(T), T)


def log10_gamma_limiting(z: int, ionic: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
    """Debye–Hückel limiting law, log₁₀ γ = −A z² √I."""
    return -A * z**2 * torch.sqrt(ionic)


def log10_gamma_extended(z: int, ionic: torch.Tensor, A: torch.Tensor, B: torch.Tensor, a: float) -> torch.Tensor:
    """Extended Debye–Hückel law, log₁₀ γ = −A z² √I / (1 + B a √I)."""
    return -A * z**2 * torch.sqrt(ionic) / (1 + B * a * torch.sqrt(ionic))


# ---------------------------------------------------------------- linearised Poisson–Boltzmann


def pb_system(h: torch.Tensor, kap: torch.Tensor, a: float, M: int) -> tuple[torch.Tensor, torch.Tensor]:
    """A_cij and b_ci for u'' = κ_c² u on grid c (potential in units of q/4πε₀ε_r, so Coulomb is 1/r).

    interior rows:  (u_{i−1} − 2u_i + u_{i+1}) / h_c² − κ_c² u_i = 0;   u_M = 0 at the outer edge
    row 0 (r = a):  Gauss's law φ'(a) = −1/a² ⇔ u'(a) − u(a)/a = −1/a, put in with a ghost point:
                    (2u₁ − 2u₀)/h² − (2/(h a)) u₀ − κ² u₀ = −2/(h a)
    h, kap: (C,) → A (C, M, M), b (C, M).
    """
    eye = torch.eye(M, device=h.device, dtype=h.dtype)
    D2 = -2 * eye + torch.diag(torch.ones(M - 1, device=h.device, dtype=h.dtype), 1)
    D2 = D2 + torch.diag(torch.ones(M - 1, device=h.device, dtype=h.dtype), -1)
    D2[0, 1] = 2.0  # ghost point u₋₁ folded back into row 0
    A = torch.einsum("c,ij->cij", 1 / h**2, D2) - torch.einsum("c,ij->cij", kap**2, eye)
    A[:, 0, 0] -= 2 / (h * a)
    b = torch.zeros(len(h), M, device=h.device, dtype=h.dtype)
    b[:, 0] = -2 / (h * a)
    return A, b


def yukawa(r: torch.Tensor, kap: torch.Tensor, a: float) -> torch.Tensor:
    """φ_ci = e^{κ_c a}/(1 + κ_c a) · e^{−κ_c r_ci}/r_ci   (units of q/4πε₀ε_r): the screened Coulomb potential."""
    k = kap[:, None]
    return torch.exp(-k * (r - a)) / ((1 + k * a) * r)


def ln_gamma_from_potential(phi_a: torch.Tensor, a: float, z: int, l_B: torch.Tensor) -> torch.Tensor:
    """ln γ = (z² l_B / 2) (φ(a) − 1/a): the energy of the ion in the potential of its own cloud.

    φ(a) − 1/a is what is left at the ion's surface after removing its own bare Coulomb potential; the factor ½
    is the charging work. With the Yukawa form this gives exactly the extended Debye–Hückel law.
    """
    return 0.5 * z**2 * l_B * (phi_a - 1 / a)


def potential_scale(T: torch.Tensor, z: int) -> torch.Tensor:
    """Volts per unit of φ̃ (nm⁻¹): z e / (4π ε₀ ε_r) · 10⁹."""
    return z * E_CHARGE / (4 * math.pi * EPS0 * permittivity_water(T)) * 1e9


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/physchem/nernst_debye_huckel/tensors.py
"""Build the tensors: physical constants, the Daniell cell, concentration and temperature batches, radial grids.

Units: potentials in V, temperatures in K, molalities in mol/kg (≈ mol/L for dilute water), lengths in nm.
"""

import torch

F = 96485.332  # Faraday constant, C/mol
R_GAS = 8.314462  # gas constant, J/(mol K)
N_A = 6.02214076e23  # Avogadro, 1/mol
E_CHARGE = 1.602176634e-19  # C
K_B = 1.380649e-23  # J/K
EPS0 = 8.8541878128e-12  # F/m

# Daniell cell  Zn | Zn²⁺(aq) || Cu²⁺(aq) | Cu:   Cu²⁺ + Zn → Cu + Zn²⁺,  two electrons, E° ≈ 1.10 V
N_ELECTRONS = 2
Z_ION = 2  # Zn²⁺ and Cu²⁺, each with SO₄²⁻: a 2:2 salt, ionic strength I = ½ Σ m z² = 4 m


def temperatures(device: torch.device) -> torch.Tensor:
    """A batch of cell temperatures, 0–80 °C   (T,) in K."""
    return torch.tensor([0.0, 20.0, 40.0, 60.0, 80.0], device=device) + 273.15


def log_ratios(device: torch.device) -> torch.Tensor:
    """log₁₀ Q of the reaction quotient Q = a(Zn²⁺)/a(Cu²⁺), from 10⁻⁴ to 10⁴   (Q,)."""
    return torch.linspace(-4.0, 4.0, 81, device=device)


def zinc_molalities(n: int, device: torch.device) -> torch.Tensor:
    """The swept Zn²⁺ molality m (C,): 10⁻⁴ … 10⁻¹ mol/kg (one animation frame each)."""
    return torch.logspace(-4.0, -1.0, n, device=device)


def ionic_strength(m: torch.Tensor, z: int = Z_ION) -> torch.Tensor:
    """I = ½ (m z² + m z²) = m z² for a symmetric z:z salt   (same shape as m)."""
    return m * z**2


def radial_grids(
    kappa: torch.Tensor, a: float, n_points: int, reach: float = 12.0
) -> tuple[torch.Tensor, torch.Tensor]:
    """r_ci (C, M) from the ion surface r = a out to a + reach/κ_c, and the spacing h_c (C,), in nm.

    Each concentration gets its own grid measured in Debye lengths, so a thick cloud and a thin cloud are
    resolved equally well. Point M (not stored) sits at the outer edge, where φ = 0.
    """
    h = reach / kappa / n_points  # (C,)
    i = torch.arange(n_points, device=kappa.device, dtype=kappa.dtype)  # (M,)
    return a + torch.einsum("c,i->ci", h, i), h


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/physchem/nernst_debye_huckel/solve.py
"""Solve the linearised Poisson–Boltzmann equation for every concentration at once with torch.linalg.solve."""

import torch

from .equation import pb_system


def solve_pb(r: torch.Tensor, h: torch.Tensor, kap: torch.Tensor, a: float) -> torch.Tensor:
    """φ_ci (C, M) on the radial grids r (C, M): solve A_cij u_cj = b_ci for each c, then φ = u / r.

    torch.linalg.solve treats the leading dimension as a batch: one call, one dense system per concentration.
    """
    A, b = pb_system(h, kap, a, r.shape[1])
    u = torch.linalg.solve(A, b)  # (C, M)
    return u / r


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/physchem/nernst_debye_huckel/plot.py
"""Animate a dilution sweep of the Zn²⁺ half-cell: EMF, activity coefficients, Debye length, the ion's cloud."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from . import equation


def animate(
    log_Q, T_batch, E_tq, m_zn, m_cu, E_ideal, E_lim, E_ext, I_zn, lg_pb, A, B, a, z, T, r, phi, kap, volts, path: Path
) -> Path:
    lq, Tb, E_tq = log_Q.cpu().numpy(), T_batch.cpu().numpy() - 273.15, E_tq.cpu().numpy()
    m, ion, lg_pb = m_zn.cpu().numpy(), I_zn.cpu().numpy(), lg_pb.cpu().numpy()
    x_ratio = np.log10(m / m_cu.item())
    dE_lim = 1e3 * (E_lim - E_ideal).cpu().numpy()  # mV
    dE_ext = 1e3 * (E_ext - E_ideal).cpu().numpy()
    E_ext_np = E_ext.cpu().numpy()
    r_np, phi_mV, kinv = r.cpu().numpy(), 1e3 * volts.item() * phi.cpu().numpy(), 1 / kap.cpu().numpy()
    V = 1e3 * volts.item()  # mV per unit of φ̃
    C = len(m)

    fig = plt.figure(figsize=(9.0, 5.4))
    gs = fig.add_gridspec(2, 3)
    ax_n, ax_d, ax_g = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[0, 2])
    ax_k, ax_p = fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1:])

    # --- (a) Nernst: E against log Q for a batch of temperatures ---
    cmap = plt.get_cmap("coolwarm")
    for t in range(len(Tb)):
        ax_n.plot(lq, E_tq[t], color=cmap(t / (len(Tb) - 1)), lw=1.3, label=f"{Tb[t]:.0f} °C")
    ax_n.plot(x_ratio, E_ext_np, "k--", lw=1.2, label=r"with γ, x = log$_{10}$(m$_{Zn}$/m$_{Cu}$)")
    (mark_n,) = ax_n.plot([], [], "o", color="#f59e0b", ms=7, zorder=5)
    ax_n.set_xlabel(r"$\log_{10}\,Q = \log_{10}(a_{Zn}/a_{Cu})$", labelpad=1)
    ax_n.set_ylabel("EMF (V)")
    ax_n.set_title("Nernst: slope 2.303RT/nF", fontsize=10)
    ax_n.legend(fontsize=6.5, loc="lower left", ncol=2)

    # --- (b) how much the activities change the EMF ---
    ax_d.semilogx(m, dE_lim, ":", color="#dc2626", lw=1.5, label="limiting law")
    ax_d.semilogx(m, dE_ext, "-", color="#2563eb", lw=1.5, label="extended law")
    ax_d.axhline(0, color="#9ca3af", lw=1)
    (mark_d,) = ax_d.plot([], [], "o", color="#f59e0b", ms=7, zorder=5)
    ax_d.set_xlabel("m(Zn²⁺)  (mol/kg)", labelpad=1)
    ax_d.set_ylabel(r"$E - E_{ideal}$ (mV)")
    ax_d.set_title(f"activity correction, m(Cu²⁺) = {m_cu.item():g}", fontsize=10)
    ax_d.legend(fontsize=7)

    # --- (c) activity coefficient of Zn²⁺ against √I ---
    s = np.linspace(0, np.sqrt(ion.max()) * 1.05, 100)
    I_s = torch.tensor(s**2, dtype=A.dtype)
    ax_g.plot(
        s, 10 ** equation.log10_gamma_limiting(z, I_s, A.cpu()).numpy(), ":", color="#dc2626", lw=1.5, label="limiting"
    )
    ax_g.plot(
        s,
        10 ** equation.log10_gamma_extended(z, I_s, A.cpu(), B.cpu(), a).numpy(),
        color="#2563eb",
        lw=1.5,
        label="extended",
    )
    (dots_g,) = ax_g.plot([], [], "o", ms=3, mfc="none", color="#16a34a", label="from PB cloud")
    (mark_g,) = ax_g.plot([], [], "o", color="#f59e0b", ms=7, zorder=5)
    ax_g.set_ylim(0, 1.05)
    ax_g.set_xlabel(r"$\sqrt{I}$  (mol/kg)$^{1/2}$", labelpad=1)
    ax_g.set_ylabel(r"$\gamma(\mathrm{Zn^{2+}})$")
    ax_g.set_title("Debye–Hückel", fontsize=10)
    ax_g.legend(fontsize=7, loc="lower left")

    # --- (d) Debye length against concentration ---
    l_11 = 1 / equation.kappa(torch.tensor(m), T.cpu()).numpy()  # a 1:1 salt, I = m
    ax_k.loglog(m, l_11, color="#9ca3af", lw=1.3, label="1:1 salt (NaCl)")
    ax_k.loglog(m, kinv, color="#2563eb", lw=1.5, label="2:2 salt (ZnSO₄)")
    (mark_k,) = ax_k.plot([], [], "o", color="#f59e0b", ms=7, zorder=5)
    ax_k.set_xlabel("molality (mol/kg)", labelpad=1)
    ax_k.set_ylabel(r"Debye length $\kappa^{-1}$ (nm)")
    ax_k.legend(fontsize=7)

    # --- (e) the potential around one Zn²⁺ ion ---
    r_view = np.linspace(a, 6.0, 300)
    ax_p.plot(r_view, V / r_view, ":", color="#6b7280", lw=1.3, label="bare Coulomb  q/4πεr")
    (line_y,) = ax_p.plot([], [], color="#2563eb", lw=1.6, label=r"Yukawa $e^{-\kappa r}/r$ (analytic)")
    (dots_p,) = ax_p.plot([], [], "o", ms=3.5, mfc="none", color="#dc2626", label="linearised PB (linalg.solve)")
    line_k = ax_p.axvline(a, color="#f59e0b", lw=1.5, ls="--")
    text_k = ax_p.text(0, 0.9, "", color="#b45309", fontsize=9, transform=ax_p.get_xaxis_transform())
    ax_p.axvspan(0, a, color="#93c5fd", alpha=0.5)
    ax_p.text(
        a / 2, 0.5, "ion", ha="center", rotation=90, fontsize=8, color="#1e40af", transform=ax_p.get_xaxis_transform()
    )
    ax_p.set_xlim(0, 6.0)
    ax_p.set_ylim(0, V / a * 1.05)
    ax_p.set_xlabel("distance from the ion centre r (nm)", labelpad=1)
    ax_p.set_ylabel("φ (mV)")
    ax_p.legend(fontsize=7.5, loc="upper right")
    title = ax_p.set_title("screened potential around Zn²⁺", fontsize=10)

    def update(c: int):
        mark_n.set_data([x_ratio[c]], [E_ext_np[c]])
        mark_d.set_data([m[c]], [dE_ext[c]])
        dots_g.set_data(np.sqrt(ion[: c + 1]), 10 ** lg_pb[: c + 1])
        mark_g.set_data([np.sqrt(ion[c])], [10 ** lg_pb[c]])
        mark_k.set_data([m[c]], [kinv[c]])
        k = 1 / kinv[c]
        line_y.set_data(r_view, V * np.exp(-k * (r_view - a)) / ((1 + k * a) * r_view))
        keep = r_np[c] <= 6.0
        stride = max(1, keep.sum() // 30)
        dots_p.set_data(r_np[c][keep][::stride], phi_mV[c][keep][::stride])
        x_k = min(a + kinv[c], 5.95)
        line_k.set_xdata([x_k, x_k])
        text_k.set_position((x_k + 0.05, 0.55))
        text_k.set_text(f"a + κ⁻¹ = {a + kinv[c]:.2f} nm")
        title.set_text(f"screened potential around Zn²⁺   ·   m = {m[c]:.1e} mol/kg, I = {ion[c]:.3g}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(C, 45), path, fps=8)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'T_C': 25.0,
    'm_Cu': 0.01,
    'E0': 1.1,
    'a_ion': 0.5,
    'n_points': 400
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.physchem.nernst_debye_huckel")]:
    del sys.modules[name]
main = importlib.import_module("problems.physchem.nernst_debye_huckel.main")
problem_dir = Path("problems/physchem/nernst_debye_huckel")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
